Cybersecurity/Fraud Detection ML Capstone Workflow

1. Import Libraries

In [26]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

2. Load Dataset

In [27]:
import pandas as pd
import glob
import os

# Folder containing CICIDS CSV files
path = r"C:\Users\bhuva\Downloads\MachineLearningCSV\MachineLearningCVE"

# Get all CSV files
files = glob.glob(os.path.join(path, "*.csv"))

# Load and combine datasets
df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

# Clean column names
df.columns = df.columns.str.strip()

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset Shape:
(5661486, 79)

Columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Co

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [28]:
df['Label'].value_counts()

Label
BENIGN                        4546194
DoS Hulk                       462146
PortScan                       317860
DDoS                           256054
DoS GoldenEye                   20586
FTP-Patator                     15876
SSH-Patator                     11794
DoS slowloris                   11592
DoS Slowhttptest                10998
Bot                              3932
Web Attack � Brute Force         3014
Web Attack � XSS                 1304
Infiltration                       72
Web Attack � Sql Injection         42
Heartbleed                         22
Name: count, dtype: int64

In [ ]:
df.to_csv(
    r"C:\Users\bhuva\Downloads\CICIDS2017_combined.csv",
    index=False
)

Dataset Overview

In [ ]:
# Dataset shape

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Column names

df.columns.tolist()

3. Exploratory Data Analysis (EDA)

Missing Values

In [ ]:
missing = df.isnull().sum()

missing[missing > 0]

Dataset Statistics

In [ ]:
df.describe()

Class Distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x='Label',
    data=df
)

plt.xticks(rotation=90)
plt.title("Network Traffic Class Distribution")

plt.show()

Target Check

In [ ]:
df['Label'].value_counts()

4. Data Preparation
Convert Target into Binary Classification

Normal traffic = 0
Attack traffic = 1

In [ ]:
df['Label'] = df['Label'].apply(
    lambda x: 0 if x == 'BENIGN' else 1
)


df['Label'].value_counts()

Define Features and Target

In [ ]:
X = df.drop('Label', axis=1)

y = df['Label']

Handle Missing / Infinite Values

In [ ]:
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


X = X.fillna(0)

Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)

5. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

6. Model Training

Random Forest Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


model.fit(
    X_train_scaled,
    y_train
)

Predictions

In [ ]:
y_pred = model.predict(
    X_test_scaled
)


y_prob = model.predict_proba(
    X_test_scaled
)[:,1]

7. Model Evaluation


Accuracy

In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

accuracy

Precision/Recall/F1

In [ ]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)


plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

ROC Curve and AUC

In [ ]:
auc = roc_auc_score(
    y_test,
    y_prob
)

print("AUC:", auc)


fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)


plt.figure(figsize=(7,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.3f}"
)


plt.plot(
    [0,1],
    [0,1],
    '--'
)


plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

8. Feature Importance

In [ ]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_

})


importance = importance.sort_values(
    by="Importance",
    ascending=False
)


importance.head(10)

Visualization

In [ ]:
plt.figure(figsize=(10,6))


sns.barplot(
    data=importance.head(10),
    x="Importance",
    y="Feature"
)


plt.title(
    "Top 10 Important Features"
)


plt.show()

9. Interpretation / Reflection

The Random Forest model was able to identify patterns between normal and malicious network traffic. 
The model achieved strong classification performance; however, accuracy alone is not enough because cybersecurity datasets often contain class imbalance.

The confusion matrix showed false positives and false negatives, which represent incorrect classifications. False negatives are especially important because they represent missed attacks.

Important features helped identify network behaviors associated with suspicious activity. Future improvements include balancing the dataset using SMOTE, hyperparameter tuning, and testing deep learning approaches.